In [1]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PayloadSchemaType, PointStruct, SparseVectorParams, Document,Prefetch, FusionQuery
from qdrant_client import models

import pandas as pd
import openai
import fastembed
import cohere

In [2]:
qdrant_client = QdrantClient("http://localhost:6333")

/Users/kishorkumarparoi/Desktop/Maven - The AI Engineering Bootcamp /Resources/AIE5-main/23_Rag_Hybrid_Re_Rank/.venv/lib/python3.12/site-packages/qdrant_client/qdrant_remote.py:290: UserWarning: Failed to obtain server version. Unable to check client-server compatibility. Set check_compatibility=False to skip version check.
  show_warning(


In [12]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Retrieve API keys from environment variables
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GEMINI_API_KEY')
qdrant_url = os.getenv('QDRANT_URL')
qdrant_api_key = os.getenv('QDRANT_API_KEY')
langsmith_api_key = os.getenv('LANGSMITH_API_KEY')
cohere_api_key = os.getenv('COHERE_API_KEY')
if qdrant_url and "qdrant:6333" in qdrant_url:
    # Docker service host is not resolvable from a local notebook kernel
    qdrant_url = qdrant_url.replace("qdrant:6333", "localhost:6333")

# Verify keys are loaded
print(f"OpenAI API Key present: {bool(openai_api_key)}")
print(f"Google API Key present: {bool(google_api_key)}")
print(f"Qdrant URL present: {bool(qdrant_url)}")
print(f"Qdrant API Key present: {bool(qdrant_api_key)}")
print(f"Langsmith API Key present: {bool(langsmith_api_key)}")
print(f"Cohere API Key present: {bool(cohere_api_key)}")

OpenAI API Key present: True
Google API Key present: False
Qdrant URL present: True
Qdrant API Key present: False
Langsmith API Key present: True
Cohere API Key present: True


In [3]:
def get_embedding(text, model= "text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding

In [4]:
def get_embeddings_batch(text_list, model= "text-embedding-3-small", batch_size=100):
    if(len(text_list) <= batch_size):
        response = openai.embeddings.create(input=text_list, model=model)
        return [embedding.embedding for embedding in response.data]
    all_embeddings = []
    counter = 1
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i+batch_size]
        response = openai.embeddings.create(input=batch, model=model)
        all_embeddings.extend([embedding.embedding for embedding in response.data])
        print(f"Processed batch {counter} / {len(text_list) // batch_size + 1}")
        counter += 1
    return all_embeddings

In [36]:
def retrieve_data(query, qdrant_client, top_k=20):
    query_embedding = get_embedding(query)
    search_result = qdrant_client.query_points(
        collection_name="Amazon_Electronics_Products",
        prefetch = [
            Prefetch(
            query = query_embedding,
            using= "text-embedding-3-small",
            limit = 20
           ),
           Prefetch(
            query = Document(text=query, model="qdrant/bm25"),
            using= "bm25",
            limit = 20
           )
        ],
        query=FusionQuery(fusion=models.Fusion.RRF),
        limit=top_k
    )

    retrieved_context_ids = []
    retrieved_contexts = []
    similarity_scores = []
    retrieved_context_ratings = []
    retrieved_context_prices = []
    retrieved_context_images = []
    retrieved_context_rating_numbers = []

    for result in search_result.points:
        retrieved_context_ids.append(result.payload['parent_asin'])
        retrieved_contexts.append(result.payload['processed_description'])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload['average_rating'])
        retrieved_context_prices.append(result.payload['price'])
        retrieved_context_images.append(result.payload['image_url'])
        retrieved_context_rating_numbers.append(result.payload['rating_number'])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_contexts": retrieved_contexts,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings,
        "retrieved_context_prices": retrieved_context_prices,
        "retrieved_context_images": retrieved_context_images,
        "retrieved_context_rating_numbers": retrieved_context_rating_numbers
    }

In [38]:
query = "What are the best wireless headphones under $100 with good battery life?"
results = retrieve_data(query, qdrant_client, top_k=20)

In [39]:
results

{'retrieved_context_ids': ['B091Q39GV6',
  'B0845ML6WD',
  'B00M95KNJW',
  'B0B37WTM91',
  'B0B8CY1MJ4',
  'B08JCB3V5V',
  'B079ZNCN64',
  'B07WF9SLQP',
  'B083L8JL1X',
  'B07YQ6LNJM',
  'B0BVLGGGBM',
  'B00BDC7890',
  'B0BBZ2JGW4',
  'B0B1ZQWF36',
  'B09YRGYD9J',
  'B09K67QXKT',
  'B09SZ7BMSX',
  'B0BKKC4JNY',
  'B0BL78QGLK',
  'B08HMDLY11'],
 'retrieved_contexts': ['iLive Truly Wire-Free Earbuds with Active Noise Canceling, Charging Case, Includes 3 Set of Ear Tips, Black (IAEBT600B) ["Say hello to true wireless freedom and goodbye to messy cords. With no strings holding you back, you can do more and be more in your sweatproof truly wireless earbuds. Block out the noise with the active noise cancelling feature. The included recharging case not only protects the earbuds while you stow them away but can also recharge your earbuds fully between your listening sessions. The right and left earbuds are paired together right out of the box, so all you have to do is connect via Bluetooth to 

Reranking

In [40]:
cohere_client = cohere.ClientV2(cohere_api_key)

In [41]:
cohere_client

In [42]:
to_rerank = results['retrieved_contexts']
to_rerank_ids = results['retrieved_context_ids']

In [43]:
to_rerank

['iLive Truly Wire-Free Earbuds with Active Noise Canceling, Charging Case, Includes 3 Set of Ear Tips, Black (IAEBT600B) ["Say hello to true wireless freedom and goodbye to messy cords. With no strings holding you back, you can do more and be more in your sweatproof truly wireless earbuds. Block out the noise with the active noise cancelling feature. The included recharging case not only protects the earbuds while you stow them away but can also recharge your earbuds fully between your listening sessions. The right and left earbuds are paired together right out of the box, so all you have to do is connect via Bluetooth to your favorite device. One key control (Bluetooth, play, pause, phone, on/off) and a waterproof design put these \'buds a cut above the rest. So, what are you waiting for? Live wire-free today. Included are 3 sets of eartips: 1 small, 1 medium, 1 large and USB-C cable."] All Electronics [\'Earbuds feature a sweatproof design and are paired together right out of the bo

In [44]:
to_rerank_ids

['B091Q39GV6',
 'B0845ML6WD',
 'B00M95KNJW',
 'B0B37WTM91',
 'B0B8CY1MJ4',
 'B08JCB3V5V',
 'B079ZNCN64',
 'B07WF9SLQP',
 'B083L8JL1X',
 'B07YQ6LNJM',
 'B0BVLGGGBM',
 'B00BDC7890',
 'B0BBZ2JGW4',
 'B0B1ZQWF36',
 'B09YRGYD9J',
 'B09K67QXKT',
 'B09SZ7BMSX',
 'B0BKKC4JNY',
 'B0BL78QGLK',
 'B08HMDLY11']

In [45]:
response = cohere_client.rerank(
    model="rerank-v4.0-pro",
    query=query,
    documents=to_rerank,
    top_n=20,
)

In [46]:
response

V2RerankResponse(id='12df98f1-9f70-4e1a-93dd-c9c65a5794ca', results=[V2RerankResponseResultsItem(index=9, relevance_score=0.8886042), V2RerankResponseResultsItem(index=14, relevance_score=0.88063294), V2RerankResponseResultsItem(index=3, relevance_score=0.878356), V2RerankResponseResultsItem(index=7, relevance_score=0.8632072), V2RerankResponseResultsItem(index=1, relevance_score=0.86041605), V2RerankResponseResultsItem(index=15, relevance_score=0.86041605), V2RerankResponseResultsItem(index=13, relevance_score=0.85709965), V2RerankResponseResultsItem(index=17, relevance_score=0.84877175), V2RerankResponseResultsItem(index=5, relevance_score=0.82927084), V2RerankResponseResultsItem(index=4, relevance_score=0.7797246), V2RerankResponseResultsItem(index=6, relevance_score=0.7783799), V2RerankResponseResultsItem(index=0, relevance_score=0.7688026), V2RerankResponseResultsItem(index=18, relevance_score=0.7688026), V2RerankResponseResultsItem(index=12, relevance_score=0.7546258), V2RerankRe

In [47]:
print(response)

id='12df98f1-9f70-4e1a-93dd-c9c65a5794ca' results=[V2RerankResponseResultsItem(index=9, relevance_score=0.8886042), V2RerankResponseResultsItem(index=14, relevance_score=0.88063294), V2RerankResponseResultsItem(index=3, relevance_score=0.878356), V2RerankResponseResultsItem(index=7, relevance_score=0.8632072), V2RerankResponseResultsItem(index=1, relevance_score=0.86041605), V2RerankResponseResultsItem(index=15, relevance_score=0.86041605), V2RerankResponseResultsItem(index=13, relevance_score=0.85709965), V2RerankResponseResultsItem(index=17, relevance_score=0.84877175), V2RerankResponseResultsItem(index=5, relevance_score=0.82927084), V2RerankResponseResultsItem(index=4, relevance_score=0.7797246), V2RerankResponseResultsItem(index=6, relevance_score=0.7783799), V2RerankResponseResultsItem(index=0, relevance_score=0.7688026), V2RerankResponseResultsItem(index=18, relevance_score=0.7688026), V2RerankResponseResultsItem(index=12, relevance_score=0.7546258), V2RerankResponseResultsItem(

In [48]:
reranked_results = [to_rerank[result.index] for result in response.results]

In [49]:
reranked_results

["JBL Reflect Contour 2 Wireless In-Ear Headphones - Black - JBLREFCONTOUR2BAM (Renewed) ['Reflective cables'] All Electronics ['JBL signature sound', '10 hour battery life with speed charge', '3-Button remote with microphone and voice assistant', 'Ergonomic ear tips with hook design', 'Reflective cables'] 28.99 ['Electronics', 'Headphones, Earbuds & Accessories', 'Headphones & Earbuds', 'Earbud Headphones']",
 'iKF Find Pro Wireless Earbuds Built-in Mic,Waterproof Bluetooth 5.0 in-Ear Headphones with Charging Case, Volume Control Stereo TWS Earphones for Sport (White)… [] All Electronics [\'【Smart in-ear sensing】-Find Pro wireless earbuds feature Built-in smart sensor chip, take off the earphones automatic pause, put on them automatic playback; It features semi-in-ear design and lightweight for more comfortable to wear.\', \'【Stereo Sound】 –iKF Find Pro Bluetooth headphones equipped with 14mm dynamic coil speaker and composite diaphragm, built-in AAC audio decoding, enjoy the professi

In [50]:
response = cohere_client.rerank(
    model="rerank-v4.0-fast",
    query=query,
    documents=to_rerank,
    top_n=20,
)

In [51]:
response

V2RerankResponse(id='637632e5-5812-4a22-8afd-64c241313652', results=[V2RerankResponseResultsItem(index=9, relevance_score=0.81411195), V2RerankResponseResultsItem(index=14, relevance_score=0.787016), V2RerankResponseResultsItem(index=5, relevance_score=0.7783799), V2RerankResponseResultsItem(index=13, relevance_score=0.7383674), V2RerankResponseResultsItem(index=7, relevance_score=0.73457676), V2RerankResponseResultsItem(index=10, relevance_score=0.7198565), V2RerankResponseResultsItem(index=3, relevance_score=0.71191186), V2RerankResponseResultsItem(index=1, relevance_score=0.7103069), V2RerankResponseResultsItem(index=15, relevance_score=0.70627147), V2RerankResponseResultsItem(index=4, relevance_score=0.69064254), V2RerankResponseResultsItem(index=6, relevance_score=0.69064254), V2RerankResponseResultsItem(index=0, relevance_score=0.6856128), V2RerankResponseResultsItem(index=17, relevance_score=0.67883813), V2RerankResponseResultsItem(index=2, relevance_score=0.6633201), V2RerankRe

In [52]:
reranked_results = [to_rerank[result.index] for result in response.results]

In [53]:
reranked_results

["JBL Reflect Contour 2 Wireless In-Ear Headphones - Black - JBLREFCONTOUR2BAM (Renewed) ['Reflective cables'] All Electronics ['JBL signature sound', '10 hour battery life with speed charge', '3-Button remote with microphone and voice assistant', 'Ergonomic ear tips with hook design', 'Reflective cables'] 28.99 ['Electronics', 'Headphones, Earbuds & Accessories', 'Headphones & Earbuds', 'Earbud Headphones']",
 'iKF Find Pro Wireless Earbuds Built-in Mic,Waterproof Bluetooth 5.0 in-Ear Headphones with Charging Case, Volume Control Stereo TWS Earphones for Sport (White)… [] All Electronics [\'【Smart in-ear sensing】-Find Pro wireless earbuds feature Built-in smart sensor chip, take off the earphones automatic pause, put on them automatic playback; It features semi-in-ear design and lightweight for more comfortable to wear.\', \'【Stereo Sound】 –iKF Find Pro Bluetooth headphones equipped with 14mm dynamic coil speaker and composite diaphragm, built-in AAC audio decoding, enjoy the professi